# 01 — Dados e linha de base

A primeira pergunta é simples: antes de intervir no modelo, ele distingue pedidos nocivos de pedidos inofensivos na direção esperada? Medimos isso pelo log-odds do primeiro token de recusa e também pelas respostas geradas.

O resultado esperado é um log-odds positivo nos exemplos nocivos e negativo nos inofensivos. Essa separação é o ponto de partida; ainda não é evidência de que encontramos um mecanismo causal.

In [ ]:
# Fixamos a raiz, a configuração e a semente antes de carregar o modelo.
from pathlib import Path
import json

from refusal_direction_replication.pipeline import run_baseline_stage

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
config = json.loads((ROOT / "configs" / "gemma_2b.json").read_text(encoding="utf-8"))
config

## Um par concreto

Pense em dois pedidos com o mesmo formato de conversa. Um representa a classe nociva — por exemplo, “forneça instruções para um ato inseguro” — e o outro é claramente inofensivo, como “explique a fotossíntese”. O texto nocivo real fica em `artifacts/`, fora do Git.

Para cada pedido, observamos os logits logo após a instrução. O score local é o log da probabilidade do token inicial de recusa dividido pela probabilidade dos demais tokens. O filtro mantém exemplos nocivos com score acima de zero e exemplos inofensivos abaixo de zero, como no código original.

In [ ]:
# Executamos a linha de base e gravamos apenas métricas agregadas no diretório versionado.
baseline = run_baseline_stage(ROOT)
baseline_results = baseline["results"]
baseline_results

In [ ]:
# Antes de olhar a média, deixamos visíveis alguns scores e decisões individuais.
individual_baseline = {
    "harmful_log_odds": baseline_results["harmful_validation"]["per_example_refusal_log_odds"][:8],
    "harmful_refusal_flags": baseline_results["harmful_validation"]["refusal_flags"][:8],
    "harmless_log_odds": baseline_results["harmless_validation"]["per_example_refusal_log_odds"][:8],
    "harmless_refusal_flags": baseline_results["harmless_validation"]["refusal_flags"][:8],
}
individual_baseline

## Leitura do resultado

A média resume a tendência, mas os valores individuais mostram se poucos casos extremos estão puxando o número. A filtragem segue o protocolo original e torna a etapa seguinte mais limpa, ao custo de restringir a população analisada.

As respostas completas ficam em `artifacts/generations/`, que não é versionado. O arquivo `results/01_baseline.json` guarda scores, flags, hashes e proveniência suficientes para auditar a execução sem publicar os prompts nocivos.